#  Finance AI Agent using OpenAI Agents SDK + Ollama

## Objective

In this notebook, we will build a **Finance AI Agent** that can answer stock market queries using:

- **OpenAI Agents SDK**
- **Ollama** running locally
- **Yahoo Finance (yfinance)** for live market data

The agent is capable of:

- Retrieving the latest stock price
- Fetching analyst recommendations
- Calling Python functions as tools automatically
- Running completely on a local LLM through Ollama

---

In [1]:
import os
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
os.environ["OPENAI_API_KEY"] = "ollama"

In [2]:
import yfinance as yf

from agents import (
    Agent,
    Runner,
    function_tool,
    set_tracing_disabled,
)
import asyncio
import gradio as gr

D:\Tech Project\Finace AI Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Disable Cloud Tracing

The OpenAI Agents SDK supports tracing through OpenAI.

Since we are running everything locally with Ollama, cloud tracing is unnecessary and is disabled.

In [3]:
# Disable OpenAI cloud tracing
set_tracing_disabled(True)

# Tool 1 — Get Current Stock Price

This tool retrieves the **latest closing price** for a stock ticker using Yahoo Finance.

### Input

Ticker Symbol

Example:

- TSLA
- AAPL
- MSFT

### Output

A formatted string containing the latest stock price.

In [4]:
# ============================================================
# Tool: Current Stock Price
# ============================================================

@function_tool
def get_stock_price(ticker: str) -> str:
    """
    Fetch the latest closing stock price
    for the given ticker symbol.
    """

    # Create Yahoo Finance object
    stock = yf.Ticker(ticker)

    # Download one day of price history
    history = stock.history(period="1d")

    # Extract latest closing price
    latest_close = history["Close"].iloc[-1]

    return f"The current price of {ticker} is ${latest_close:.2f}"

# Tool 2 — Analyst Recommendations

This tool retrieves analyst ratings from Yahoo Finance.

If recommendations are available, the latest 10 entries are returned.

Otherwise, a suitable message is displayed.

In [5]:
# ============================================================
# Tool: Analyst Recommendations
# ============================================================

@function_tool
def get_analyst_recommendations(ticker: str) -> str:
    """
    Retrieve analyst recommendations
    for the specified stock ticker.
    """

    # Create Yahoo Finance object
    stock = yf.Ticker(ticker)

    # Download analyst recommendation data
    recommendations = stock.recommendations

    # Handle missing recommendation data
    if recommendations is None or recommendations.empty:
        return f"No analyst recommendations found for {ticker}"

    # Return the latest 10 recommendations
    latest_recommendations = recommendations.tail(10)
    
    #return latest_recommendations.to_string()

    return latest_recommendations.to_markdown()


# Create the Finance Agent

The Finance Agent is responsible for:

- Understanding user queries
- Deciding when to use tools
- Calling the appropriate function
- Returning a natural language response

### Model

```
gemma4:31b-cloud
```

### Available Tools

- Current Stock Price
- Analyst Recommendations

In [6]:
import yfinance as yf
stock = yf.Ticker("TSLA")
print(stock.news[:1])  # inspect raw structure

[{'id': '7b99391e-9228-43c1-b448-6e9d8d43b527', 'content': {'id': '7b99391e-9228-43c1-b448-6e9d8d43b527', 'contentType': 'VIDEO', 'title': 'YieldMax launches a new SpaceX options ETF: What investors should know', 'description': '<p>SpaceX (<a target="_blank" rel="" class="link" href="https://finance.yahoo.com/quote/SPCX/" data-i13n="slk:SPCX;cpos:1;pos:1">SPCX</a>) dropped below its IPO price on Wednesday, sinking below $140 per share.</p>\n<p>YieldMax ETFs chief strategist Mike Khouw comes on Market Domination to discuss his firm\'s new ETF product with options-based strategies on SpaceX: the YieldMax SPCX Option Income Strategy ETF (<a target="_blank" rel="" class="link" href="https://finance.yahoo.com/quote/YSPC/" data-i13n="cpos:2;pos:1">YSPC</a>).</p>', 'summary': "SpaceX (SPCX) dropped below its IPO price on Wednesday, sinking below $140 per share. YieldMax ETFs chief strategist Mike Khouw comes on Market Domination to discuss his firm's new ETF product with options-based strateg

In [7]:
# ============================================================
# Create the Finance AI Agent
# ============================================================

@function_tool
def get_stock_news(ticker: str) -> str:
    """
    Fetch latest news headlines and summaries for the given stock ticker.
    """
    stock = yf.Ticker(ticker)
    news_items = stock.news

    if not news_items:
        return f"No recent news found for {ticker}"

    articles = []
    for item in news_items[:10]:
        content = item.get("content", {})

        title = content.get("title", "No title")
        summary = content.get("summary", "")
        provider = content.get("provider", {})
        publisher = provider.get("displayName", "Unknown source")

        if title == "No title":
            continue  # skip malformed entries

        articles.append(f"- [{publisher}] {title}\n  {summary}")

    if not articles:
        return f"No recent news found for {ticker}"

    return "\n".join(articles)

# Run the Agent

We now send a user query to the agent.

The agent will:

1. Understand the question
2. Decide whether a tool is required
3. Execute the tool
4. Generate the final response

In [8]:
# ============================================================
# Execute the Agent
# ============================================================

finance_agent = Agent(
    name="Finance Agent",
    instructions=(
        "You are a helpful finance assistant. "
        "Whenever appropriate, present structured information using tables."
    ),
    model="gemma4:31b-cloud",
    tools=[get_stock_price, get_analyst_recommendations],
)

news_sentiment_agent = Agent(
    name="News Sentiment Agent",
    instructions=(
        "You ALWAYS analyze stock news sentiment immediately — never ask for permission or confirmation. "
        "When given any query mentioning a company or ticker, immediately call get_stock_news for that ticker. "
        "Do not ask the user if they want news analysis — just do it. "
        "Steps: "
        "1. Call get_stock_news with the ticker extracted from the query. "
        "2. Judge overall tone: Positive, Negative, or Mixed/Neutral. "
        "3. Give a short summary (3-5 bullet points) in simple, plain English. "
        "4. End with one line: 'Overall Sentiment: <Positive/Negative/Mixed>'. "
        "Never respond with a question. Always respond with the analysis."
    ),
    model="gemma4:31b-cloud",
    tools=[get_stock_news],
)

# Display the Response

The final output generated by the AI agent is displayed below.

In [9]:
# Print the final response

#print(result.final_output)

In [10]:
async def run_both_agents(user_input: str):
    price_task = Runner.run(finance_agent, user_input)
    sentiment_task = Runner.run(news_sentiment_agent, user_input)

    price_result, sentiment_result = await asyncio.gather(price_task, sentiment_task)

    return price_result.final_output, sentiment_result.final_output

In [11]:
def chat_with_agents(user_input: str):
    price_output, sentiment_output = asyncio.run(run_both_agents(user_input))
    return price_output, sentiment_output

In [12]:
demo = gr.Interface(
    fn=chat_with_agents,
    inputs=gr.Textbox(label="Ask about a stock", placeholder="e.g. What is the current price of TSLA?"),
    outputs=[
        gr.Textbox(label="Stock Price / Recommendations", lines=10),
        gr.Textbox(label="News Sentiment", lines=10),
    ],
    title="Finance AI Agent",
    description="Get stock price info and public sentiment side by side.",
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [13]:
'''
if __name__ == "__main__":
    demo.launch()
'''

'\nif __name__ == "__main__":\n    demo.launch()\n'